In [29]:
def raw_to_adata(sample, folder):
    import pandas as pd
    from shapely.geometry import MultiPoint
    import anndata as ad

    transcripts = pd.read_parquet(folder / sample / "transcripts.parquet")
    boundaries = pd.read_parquet(folder / sample / "nucleus_boundaries.parquet")

    #keeping only transcripts that were assigned to a cell
    transcripts = transcripts[transcripts['EntityID'].notna()] 
    #removing all transcripts that were assigned to cell that has less than 2 trasncripts assigned to it
    cell_counts = transcripts['EntityID'].value_counts()
    valid_cells = cell_counts[cell_counts >= 2].index
    transcripts = transcripts[transcripts['EntityID'].isin(valid_cells)]

    cell_gene_matrix = pd.crosstab(transcripts['EntityID'], transcripts['gene'])

    merged_cell_gene_matrix = boundaries.join(cell_gene_matrix, on='EntityID') #joining matrix with centorids

    # cell metadata
    obs = merged_cell_gene_matrix[['centroid_x', 'centroid_y']]

    # gene expression matrix
    X = merged_cell_gene_matrix.drop(columns=['centroid_x', 'centroid_y', 'geometry'])
    X = X.fillna(0)
    # gene metadata
    var = pd.DataFrame(index=X.columns)

    # create AnnData object
    adata = ad.AnnData(X=X, obs=obs, var=var)

    #final touches needed for Segtraq compatibiltiy
    adata.obs["cell_id"] = adata.obs.index.astype(int)  # segger_cell_id is actually INDEX here, and segtraq exepcts a column called cell_id not segger_cell_id
    adata.obs["region"] = "cell_boundaries"
    adata.obs["region"] = adata.obs["region"].astype("category")

    # add spatial coordinates to obsm
    adata.obsm['spatial'] = obs[['centroid_x', 'centroid_y']].values
    
    adata.write_h5ad(folder / sample / "adata.h5ad")

In [ ]:
import os
from pathlib import Path

# --- CONFIGURATION: fill in paths before running ---
INPUT_DIR = Path("")  # folder containing per-sample subdirs with transcripts and boundaries

for folder_name in os.listdir(INPUT_DIR):
    sample_folder_path = INPUT_DIR / folder_name
    #print(sample_folder_path)
    print(folder_name)
    raw_to_adata(folder_name, INPUT_DIR)


In [102]:
def adata_to_spatialdata(sample, input_folder):
    import spatialdata as sd
    import anndata as ad
    import pandas as pd
    import geopandas as gpd

    #loading the anndata object and the transcripts dataframe
    adata = ad.read_h5ad(f'{input_folder}/{sample}/adata.h5ad')
    transcripts_df = pd.read_parquet(f'{input_folder}/{sample}/transcripts.parquet')
    cell_shapes_gdf = gpd.read_parquet(f'{input_folder}/{sample}/nucleus_boundaries.parquet')
    cell_shapes_gdf["cell_id"] = cell_shapes_gdf["EntityID"].astype(str)
    cell_shapes_gdf['EntityID'] = cell_shapes_gdf.set_index('EntityID').values.astype(str) #segtraq needs adata and cell_shapes_gdf to have the same index, which is cell_id, so we need to make sure they match. This line is just a check, it should return True for all cells.
    
    #transcripts file adjustements
    transcripts_df = transcripts_df.rename(columns={
        "gene": "feature_name",
        "global_x": "x",
        "global_y": "y",
        "global_z": "z"
        
    })

    transcripts_df["feature_name"] = transcripts_df["feature_name"].astype("category")
    transcripts_df['cell_id'] = transcripts_df['EntityID'].astype(str) 
    transcripts_df["EntityID"] = transcripts_df["EntityID"] .astype(str) 
    #adata adjustements

    #print(cell_shapes_gdf.head())
    

    #print(f'Number of cells in adata: {len(adata.obs)}')
    #print(f'Number of cells in cell_shapes_gdf: {len(cell_shapes_gdf)}')  
    adata.obs["cell_id"] = cell_shapes_gdf.cell_id.values.astype(str) #segtraq needs adata and cell_shapes_gdf to have the same index, which is cell_id, so we need to make sure they match. This line is just a check, it should return True for all cells.
    cell_shapes_gdf.index = cell_shapes_gdf.cell_id.values.astype(str)
    #print(adata.obs.dtypes)
    #rint(cell_shapes_gdf.dtypes)
    adata.obs_names = adata.obs["cell_id"]
    
    #NOVO
    cell_shapes_gdf = cell_shapes_gdf.set_index("cell_id")
    adata.obs["cell_id"] = adata.obs["cell_id"].astype(str)
    adata.obs_names = adata.obs["cell_id"]

    #creating the spatialdata object
    sdata = sd.SpatialData(
        points={
            "transcripts": sd.models.PointsModel.parse(transcripts_df)
        },
        shapes={
            "cell_boundaries": sd.models.ShapesModel.parse(cell_shapes_gdf)
        },
        tables={
            "table": sd.models.TableModel.parse(
                adata,
                region_key="region",
                region="cell_boundaries",
                instance_key="cell_id"
            )
        },
    )

    sdata.write(f"{input_folder}/{sample}/raw_spatialdata.zarr", overwrite=True)

In [ ]:
for folder_name in os.listdir(INPUT_DIR):
    sample_folder_path = INPUT_DIR / folder_name
    #print(sample_folder_path)
    print(folder_name)
    adata_to_spatialdata(folder_name, INPUT_DIR)
